<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/03_Advanced/L38_Tokenizer_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L38: Tokenizer Training - Custom BPE and WordPiece

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Advanced  
**Lesson:** 38 of 45

---

## Learning Objectives

By the end of this lesson, you will:
- Train a BPE tokenizer on a custom corpus using tokenizers library
- Train a WordPiece tokenizer and compare vocabulary
- Save and load custom tokenizers for use with HuggingFace models

---

## Concept: Tokenizer Training

**BPE** and **WordPiece** are subword algorithms. You provide a corpus and vocab size (or merge rules); the trainer learns merges (BPE) or word pieces (WordPiece). Domain-specific tokenizers can reduce sequence length and improve quality.

---

In [ ]:
!pip install tokenizers datasets -q
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Train BPE Tokenizer

---

In [ ]:
def get_training_corpus(ds, key="text", batch_size=1000):
    for i in range(0, len(ds), batch_size):
        yield ds[i:i+batch_size][key]

ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train", trust_remote_code=True)
ds = ds.filter(lambda x: len(x["text"].strip()) > 0)

tokenizer_bpe = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer_bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_special_tokens=False)
trainer_bpe = trainers.BpeTrainer(vocab_size=8000, special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]"])
tokenizer_bpe.train_from_iterator(get_training_corpus(ds), trainer=trainer_bpe)
tokenizer_bpe.decoder = decoders.ByteLevel()
print("BPE tokenizer trained. Vocab size:", tokenizer_bpe.get_vocab_size())

## Part 2: Encode and Decode

---

In [ ]:
enc = tokenizer_bpe.encode("Hello world. This is a test.")
print("Ids:", enc.ids[:20])
print("Decode:", tokenizer_bpe.decode(enc.ids))

## Part 3: Save and Load

---

In [ ]:
tokenizer_bpe.save("./tokenizer_bpe_l38.json")
loaded = Tokenizer.from_file("./tokenizer_bpe_l38.json")
print("Saved and loaded. Use with HuggingFace: AutoTokenizer.from_pretrained('path') after wrapping.")

## Exercises

1. Train with vocab_size=4000 and 16000; compare token counts on a fixed corpus.
2. Train a WordPiece tokenizer (models.WordPiece) and compare to BPE.
3. Train on domain-specific text (e.g. code, medical) and measure compression ratio.

---

## Key Takeaways

1. BPE: merge pairs by frequency; WordPiece: likelihood-based splits.
2. Train on the same domain as your LM for best results.
3. Save tokenizer and use with transformers (e.g. PreTrainedTokenizerFast).

---

## Next Lesson

**L39: Model Parallelism** – Pipeline and tensor parallelism.

---